# Borzoi Regulatory Sequence Prediction

![Borzoi Regulatory Sequence Prediction](https://proto-bio.github.io/proto-assets/images/tool/borzoi/hero.png)

Borzoi is a long-context regulatory sequence model published by Linder et al. (2024) that predicts regulatory signals from 524,288 bp DNA input windows. It extends the Enformer architecture with larger context, higher resolution output (6,144 bins), and replicate-specific checkpoints trained on different data splits. The tool provides two functions: single-replicate prediction for fast inference with one checkpoint, and ensemble prediction that runs all four replicates for robust consensus analysis with uncertainty estimates.

This notebook demonstrates both workflows, showing how to extract selected output tracks and compute cross-replicate statistics.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("borzoi")
display_overview("borzoi")
display_docs_section("borzoi", "Background")

## Available tools

In [ ]:
display_available_tools("borzoi")

### `run_borzoi`

This function runs a single Borzoi replicate checkpoint on a 524,288 bp DNA sequence and returns a prediction matrix of shape [num_tracks, 6144]. It is the fastest option for regulatory activity prediction when cross-replicate uncertainty estimates are not required. The replicate index (0 through 3) and species (human or mouse) can be configured independently, and output tracks can be averaged into a single signal for compact downstream analysis.

In [ ]:
import numpy as np
from proto_tools import (
    BORZOI_CONTEXT,
    BorzoiConfig,
    BorzoiInput,
    run_borzoi,
)

In [ ]:
# Display input docs
display_api_reference("borzoi", "input", "run_borzoi")

from proto_tools import NCBIEfetchConfig, NCBIEfetchInput, run_ncbi_efetch

# Fetch a real human genomic window on chr11 centred on the HBB locus, exactly the
# length Borzoi requires, instead of a synthetic repeat.
_center = 5227430  # HBB midpoint, GRCh38 chr11 (NC_000011.10)
_fetch = run_ncbi_efetch(
    NCBIEfetchInput(
        db="nuccore", identifier="NC_000011.10", return_format="fasta",
        seq_start=_center - BORZOI_CONTEXT // 2, seq_stop=_center + BORZOI_CONTEXT // 2,
    ),
    NCBIEfetchConfig(),
)
sequence = _fetch.fasta_records[0].sequence.upper()[:BORZOI_CONTEXT]
inputs = BorzoiInput(sequences=[sequence])

In [ ]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi")

single_config = BorzoiConfig(
    output_tracks=[0, 1, 2],
    species="human",
    replicate="0",
    avg_output_tracks=False,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

In [ ]:
# Run the single-replicate prediction function
single_result = run_borzoi(inputs, single_config)

In [ ]:
display_api_reference("borzoi", "output", "run_borzoi")

print(f"Tracks: {len(single_result.results[0].prediction)}")
print(f"Bins per track: {len(single_result.results[0].prediction[0])}")

### `run_borzoi_ensemble`

This function runs all four Borzoi replicate checkpoints and stacks the outputs into a tensor of shape [4, num_tracks, 6144]. Access to cross-replicate mean and standard deviation provides a measure of prediction confidence: regions where replicates agree strongly (low standard deviation) are more likely to reflect genuine regulatory signal than regions with high variance. When `avg_output_tracks` is enabled, each replicate's tracks are averaged before stacking, giving a final shape of [4, 1, 6144].

In [ ]:
from proto_tools import (
    BorzoiEnsembleConfig,
    run_borzoi_ensemble,
)

In [ ]:
# Display input docs (same BorzoiInput as single-replicate prediction)
display_api_reference("borzoi", "input", "run_borzoi_ensemble")

In [ ]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi_ensemble")

ensemble_config = BorzoiEnsembleConfig(
    output_tracks=[0, 1, 2],
    species="human",
    avg_output_tracks=True,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

In [ ]:
# Run the ensemble prediction function (runs all 4 replicates)
ensemble_result = run_borzoi_ensemble(inputs, ensemble_config)

In [ ]:
display_api_reference("borzoi", "output", "run_borzoi_ensemble")

ensemble_array = np.array(ensemble_result.results[0].predictions)
mean_pred = ensemble_array.mean(axis=0)
std_pred = ensemble_array.std(axis=0)
print(f"Replicates: {ensemble_array.shape[0]}")
print(f"Mean shape: {mean_pred.shape}")
print(f"Std max: {std_pred.max():.4f}")

### `run_borzoi_gradient`

The gradient tool turns Borzoi's track predictions into a differentiable objective. Each term names an output track, a span of 32 bp output bins, a direction, and a weight; the terms are signed so the summed objective is always minimized, and the gradient comes back with respect to a relaxed distribution over the bases you are designing.

Borzoi reads a fixed 524,288 bp context, but a design problem usually varies only a small part of it. A candidate therefore bundles the **whole** context sequence with the 1-indexed `positions` whose bases are relaxed and a `(len(positions), 4)` logits matrix aligned to them — so the gradient covers only the designed positions while the surrounding context still reaches the model.

In [ ]:
from proto_tools import (
    BorzoiGradientCandidate, BorzoiGradientConfig, BorzoiGradientInput,
    BorzoiGradientLossTerm, run_borzoi_gradient,
)

display_api_reference("borzoi", "input", "run_borzoi_gradient")

# Design a 24 bp stretch at the centre of the same HBB window used above, where
# Borzoi's output bins are densest. Positions are 1-indexed and inclusive.
design_positions = [BORZOI_CONTEXT // 2 + offset for offset in range(24)]
candidate = BorzoiGradientCandidate(
    sequence=sequence,
    positions=design_positions,
    # Start from a flat distribution over A, C, G, T at each designed position.
    logits=[[0.25, 0.25, 0.25, 0.25]] * len(design_positions),
)


In [ ]:
display_api_reference("borzoi", "config", "run_borzoi_gradient")

# Push track 0 up over the output bins covering the designed window.
config = BorzoiGradientConfig(
    loss_terms=[BorzoiGradientLossTerm(track=0, bin_start=3060, bin_end=3085, direction="max")],
)

gradient = run_borzoi_gradient(BorzoiGradientInput(inputs=candidate), config)


In [ ]:
display_api_reference("borzoi", "output", "run_borzoi_gradient")

# One result per candidate; this call passed one.
result = gradient.results[0]
print(f"Weighted objective: {result.loss:.4f}")
print(f"Designed positions: {result.metrics['designed_positions']}")
print(f"Gradient shape:     ({len(result.gradient)}, {len(result.gradient[0])})")
for term in result.metrics["loss_terms"]:
    print(f"  track {term['track']} bins {term['bin_start']}-{term['bin_end']}: "
          f"mean activity {term['mean_activity']:.4f} ({term['direction']})")
display_api_reference("borzoi", "input", "run_borzoi_gradient")
display_api_reference("borzoi", "config", "run_borzoi_gradient")
display_api_reference("borzoi", "output", "run_borzoi_gradient")


## Export Results

Single-replicate and ensemble outputs can be saved to standard file formats for downstream analysis. Supported formats include JSON and CSV.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

single_result.export(name="borzoi_single", export_path=output_dir, file_format="json")
print(f"Single-replicate result exported to {output_dir}/borzoi_single.json")

ensemble_result.export(name="borzoi_ensemble", export_path=output_dir, file_format="csv")
print(f"Ensemble summary exported to {output_dir}/borzoi_ensemble.csv")